In [11]:
import numpy as np
import pandas as pd


# =====================================================================
# 1. DATA LOADING & SPLITTING
# =====================================================================
def load_and_split_data(
    url: str, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Loads a CSV from a URL and performs a 60/20/20 train/val/test split."""
    df = pd.read_csv(url)

    # Shuffle index deterministically
    np.random.seed(seed)
    n = len(df)
    idx = np.arange(n)
    np.random.shuffle(idx)

    # Compute split boundaries
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    # Split dataframe
    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train : n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val :]].reset_index(drop=True)

    return df_train, df_val, df_test


# =====================================================================
# 2. CATEGORICAL ONE-HOT ENCODING (LEAK-FREE)
# =====================================================================
def fit_categorical_encoder(
    df_train: pd.DataFrame, cat_cols: list[str]
) -> dict[str, list]:
    """Learns unique categories for each categorical column exclusively from the training set."""
    encoding_map = {}
    for col in cat_cols:
        # Get unique non-null categories sorted for deterministic output
        unique_cats = sorted(df_train[col].dropna().unique().tolist())
        encoding_map[col] = unique_cats
    return encoding_map


def transform_categorical(
    df: pd.DataFrame, encoding_map: dict[str, list]
) -> pd.DataFrame:
    """Transforms categorical columns into one-hot binary flags using pre-learned categories."""
    encoded_dfs = []

    for col, categories in encoding_map.items():
        col_series = df[col]
        # Create a boolean/numeric binary flag for each learned category
        for cat in categories:
            feature_name = f"{col}_{cat}"
            encoded_dfs.append((col_series == cat).astype(int).rename(feature_name))

    if encoded_dfs:
        return pd.concat(encoded_dfs, axis=1)
    return pd.DataFrame(index=df.index)


# =====================================================================
# 3. MISSING VALUE IMPUTATION (LEAK-FREE)
# =====================================================================
def fit_imputer(
    df_train: pd.DataFrame, num_cols: list[str], strategy: str = "median"
) -> dict[str, float]:
    """Computes summary statistics (mean or median) for numerical columns on the training set."""
    imputation_values = {}
    for col in num_cols:
        if strategy == "median":
            val = df_train[col].median()
        elif strategy == "mean":
            val = df_train[col].mean()
        else:
            raise ValueError(f"Unsupported strategy: {strategy}")
        imputation_values[col] = float(val)
    return imputation_values


def transform_imputer(
    df: pd.DataFrame, imputation_values: dict[str, float]
) -> pd.DataFrame:
    """Imputes missing values in numerical columns using pre-computed statistics."""
    df_num = df[list(imputation_values.keys())].copy()
    return df_num.fillna(imputation_values)


# =====================================================================
# 4. OPTIONAL EXTENSIBILITY STEP (FEATURE SCALING / EXPANSION)
# =====================================================================
def fit_transform_scaler(
    df_train_num: pd.DataFrame,
    df_val_num: pd.DataFrame,
    df_test_num: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Example extension function: Standardizes numerical features (zero mean, unit variance)."""
    means = df_train_num.mean()
    stds = df_train_num.std().replace(0, 1.0)  # Prevent division by zero

    train_scaled = (df_train_num - means) / stds
    val_scaled = (df_val_num - means) / stds
    test_scaled = (df_test_num - means) / stds

    return train_scaled, val_scaled, test_scaled


# =====================================================================
# 5. FEATURE MATRIX ASSEMBLY
# =====================================================================
def prepare_feature_matrix(
    df_num: pd.DataFrame, df_cat_encoded: pd.DataFrame, add_bias: bool = True
) -> np.ndarray:
    """Combines processed numerical and categorical features into a single NumPy array with a bias column."""
    features = [df_num.values, df_cat_encoded.values]

    if add_bias:
        bias_column = np.ones((len(df_num), 1))
        features.insert(0, bias_column)

    return np.hstack(features)


# =====================================================================
# 6. RIDGE REGRESSION SOLVER (PURE NUMPY)
# =====================================================================
def train_ridge_regression(
    X: np.ndarray, y: np.ndarray, r: float = 0.0
) -> np.ndarray:
    """Fits Ridge Regression using Normal Equations: w = (X^T X + r*I)^(-1) X^T y.

    Excludes the bias column (index 0) from regularization.
    """
    n_features = X.shape[1]
    XTX = X.T @ X

    # Create Identity matrix and zero out the bias penalty
    I = np.eye(n_features)
    I[0, 0] = 0.0

    # Regularized normal equation
    XTX_reg = XTX + r * I
    weights = np.linalg.inv(XTX_reg) @ X.T @ y

    return weights


def predict(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Computes predictions given feature matrix X and weight vector."""
    return X @ weights


# =====================================================================
# 7. EVALUATION & EXPERIMENTS
# =====================================================================
def calculate_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Computes Root Mean Squared Error (RMSE)."""
    error = y_true - y_pred
    mse = np.mean(error**2)
    return float(np.sqrt(mse))


def evaluate_regularization(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    r_values: list[float],
) -> dict[float, float]:
    """Evaluates validation RMSE across multiple regularization parameters."""
    results = {}
    for r in r_values:
        w = train_ridge_regression(X_train, y_train, r=r)
        y_pred_val = predict(X_val, w)
        rmse = calculate_rmse(y_val, y_pred_val)
        results[r] = rmse
    return results


def run_seed_experiment(
    url: str,
    target_col: str,
    cat_cols: list[str],
    num_cols: list[str],
    seeds: list[int],
    r: float = 0.0,
) -> tuple[float, float, list[float]]:
    """Runs the complete end-to-end pipeline over multiple random seeds to measure stability."""
    rmse_scores = []

    for seed in seeds:
        # 1. Load and Split
        df_train, df_val, df_test = load_and_split_data(url, seed=seed)

        # Separate targets
        y_train = df_train[target_col].values
        y_val = df_val[target_col].values

        # 2. Fit and Transform Imputer
        imp_map = fit_imputer(df_train, num_cols)
        X_train_num = transform_imputer(df_train, imp_map)
        X_val_num = transform_imputer(df_val, imp_map)

        # 3. Fit and Transform One-Hot Encoder
        enc_map = fit_categorical_encoder(df_train, cat_cols)
        X_train_cat = transform_categorical(df_train, enc_map)
        X_val_cat = transform_categorical(df_val, enc_map)

        # 4. Assemble Feature Matrices
        X_train = prepare_feature_matrix(X_train_num, X_train_cat)
        X_val = prepare_feature_matrix(X_val_num, X_val_cat)

        # 5. Train and Evaluate
        weights = train_ridge_regression(X_train, y_train, r=r)
        y_pred = predict(X_val, weights)
        rmse = calculate_rmse(y_val, y_pred)

        rmse_scores.append(rmse)

    mean_rmse = float(np.mean(rmse_scores))
    std_rmse = float(np.std(rmse_scores))

    return mean_rmse, std_rmse, rmse_scores


# =====================================================================
# MAIN PIPELINE EXECUTION
# =====================================================================
if __name__ == "__main__":
    DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

    # Quick inspection to discover columns
    df_raw = pd.read_csv(DATA_URL)

    # 1. Identify Target and Features
    # Assuming 'fuel_efficiency' (or similar column in dataset) as target
    possible_targets = [
        c
        for c in df_raw.columns
        if "efficiency" in c.lower()
        or "mpg" in c.lower()
        or "target" in c.lower()
    ]
    TARGET_COL = (
        possible_targets[0] if possible_targets else df_raw.columns[-1]
    )

    feature_cols = [c for c in df_raw.columns if c != TARGET_COL]
    CAT_COLS = df_raw[feature_cols].select_dtypes(include=["str", "category"]).columns.tolist()
    NUM_COLS = df_raw[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

    print(f"Target Variable: '{TARGET_COL}'")
    print(f"Numerical Features ({len(NUM_COLS)}): {NUM_COLS}")
    print(f"Categorical Features ({len(CAT_COLS)}): {CAT_COLS}\n")

    # 2. Load & Split (Primary Run - Seed 42)
    df_train, df_val, df_test = load_and_split_data(DATA_URL, seed=42)

    # Extract Target Vectors
    y_train = df_train[TARGET_COL].values
    y_val = df_val[TARGET_COL].values
    y_test = df_test[TARGET_COL].values

    # 3. Fit/Transform Preprocessing (Train Leakage-Free)
    imputation_map = fit_imputer(df_train, NUM_COLS, strategy="median")
    df_train_num = transform_imputer(df_train, imputation_map)
    df_val_num = transform_imputer(df_val, imputation_map)
    df_test_num = transform_imputer(df_test, imputation_map)

    encoding_map = fit_categorical_encoder(df_train, CAT_COLS)
    df_train_cat = transform_categorical(df_train, encoding_map)
    df_val_cat = transform_categorical(df_val, encoding_map)
    df_test_cat = transform_categorical(df_test, encoding_map)

    # 4. Assemble Design Matrices (X)
    X_train = prepare_feature_matrix(df_train_num, df_train_cat)
    X_val = prepare_feature_matrix(df_val_num, df_val_cat)
    X_test = prepare_feature_matrix(df_test_num, df_test_cat)

    print(f"X_train shape: {X_train.shape}")
    print(f"X_val shape:   {X_val.shape}")
    print(f"X_test shape:  {X_test.shape}\n")

    # 5. Regularization Tuning Experiment
    r_candidates = [0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    val_results = evaluate_regularization(
        X_train, y_train, X_val, y_val, r_candidates
    )

    print("--- Regularization Search (Validation RMSE) ---")
    best_r = 0.0
    best_val_rmse = float("inf")

    for r_val, rmse in val_results.items():
        print(f"r = {r_val:<7} | Val RMSE = {rmse:.5f}")
        if rmse < best_val_rmse:
            best_val_rmse = rmse
            best_r = r_val

    print(f"\nBest Regularization Parameter: r = {best_r}")

    # 6. Final Evaluation on Holdout Test Set
    # Combine Train + Val to train final weights using optimal r
    X_full_train = np.vstack([X_train, X_val])
    y_full_train = np.concatenate([y_train, y_val])

    final_weights = train_ridge_regression(X_full_train, y_full_train, r=best_r)
    y_test_pred = predict(X_test, final_weights)
    test_rmse = calculate_rmse(y_test, y_test_pred)

    print(f"Final Test Set RMSE (r={best_r}): {test_rmse:.5f}\n")

    # 7. Seed Stability Experiment
seeds_to_test = [10, 20, 30, 42, 100, 2026]

mean_rmse, std_rmse, rmse_per_seed = run_seed_experiment(
    url=DATA_URL,
    target_col=TARGET_COL,
    cat_cols=CAT_COLS,
    num_cols=NUM_COLS,
    seeds=seeds_to_test,
    r=best_r,
)

print("--- Multi-Seed Stability Experiment ---")
print(f"Tested Seeds: {seeds_to_test}")

for seed, rmse in zip(seeds_to_test, rmse_per_seed):
    print(f"  seed={seed:>5d}  RMSE={rmse:.5f}")

print(f"Mean Validation RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

Target Variable: 'fuel_efficiency_mpg'
Numerical Features (7): ['model_year', 'num_doors', 'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight', 'acceleration']
Categorical Features (3): ['origin', 'fuel_type', 'drivetrain']

X_train shape: (6000, 17)
X_val shape:   (2000, 17)
X_test shape:  (2000, 17)

--- Regularization Search (Validation RMSE) ---
r = 0.0     | Val RMSE = 164.68073
r = 0.0001  | Val RMSE = 1.62883
r = 0.001   | Val RMSE = 1.62883
r = 0.01    | Val RMSE = 1.62883
r = 0.1     | Val RMSE = 1.62883
r = 1.0     | Val RMSE = 1.62882
r = 10.0    | Val RMSE = 1.62883
r = 100.0   | Val RMSE = 1.63231

Best Regularization Parameter: r = 1.0
Final Test Set RMSE (r=1.0): 1.63990

--- Multi-Seed Stability Experiment ---
Tested Seeds: [10, 20, 30, 42, 100, 2026]
  seed=   10  RMSE=1.57299
  seed=   20  RMSE=1.66372
  seed=   30  RMSE=1.60384
  seed=   42  RMSE=1.62882
  seed=  100  RMSE=1.61994
  seed= 2026  RMSE=1.58767
Mean Validation RMSE: 1.61283 ± 0.02943
